# Stage 2 · resume dense Lego through update 2,000
Start a fresh Kaggle notebook with **T4 ×2**. Attach the same private Stage 0 code/data kit and the reviewed `lego-dense-vanilla-s42-first-stop.zip` as a private input. Both zipped and expanded inputs work. Run the three code cells. The trainer restores the model, Adam and RNG states on GPU 0 and performs 1,500 more updates. At the measured 6.12 updates/s these updates take about **4 min 05 s**, plus dataset verification, caching, diagnostics and export. The 50,000-update training schedule stays fixed.

Download **`lego-dense-vanilla-s42-step2000.zip`** from Output and send it for review, including if training stops early. If an interrupted session produced that ZIP, you can attach it alongside the two original inputs in a fresh session. Within the same session, rerunning the last cell continues from the most recent saved checkpoint.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, time, zipfile
import torch

inputs = Path("/kaggle/input")
work = Path("/kaggle/working/recovery-main")
relative_run = Path("runs/lego/dense-vanilla/seed-42")
run = work / relative_run
manifest = work / "manifests/lego.json"
review = work.parent / "lego-dense-vanilla-s42-step2000.zip"
target_step = 2000
source_id = "08ff8d2fceda43759554aa6470054853"
source_digest = "0730a3442af623680fa2fe705a5e1dabbe613231238f3840ccc075cb9df66e5b"
data_hash = "bdc22418467720a3dbf5555318532caf17c250a05433f8553ae58372f11e89e2"
code_hash = "4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5"

def mounted(marker, filename, optional=False):
    roots = [p.parent for p in inputs.rglob(marker)]
    archives = list(inputs.rglob(filename))
    if not roots and not archives and optional:
        return None
    if len(roots) + len(archives) != 1:
        raise ValueError(f"Attach one copy of {filename} as a private input.")
    if roots:
        return roots[0]
    root = Path(tempfile.mkdtemp(prefix="nerf-stage2-"))
    with zipfile.ZipFile(archives[0]) as z:
        if marker not in z.namelist() or any(not (root / n).resolve().is_relative_to(root) for n in z.namelist()):
            raise ValueError(f"Invalid archive layout in {filename}")
        z.extractall(root)
    return root

repo = mounted("RECOVERY_STAGE0.txt", "stage0-input.zip")
first = mounted("stage1-review.json", "lego-dense-vanilla-s42-first-stop.zip")
partial = mounted("stage2-review.json", review.name, optional=True)
work.mkdir(parents=True, exist_ok=True)
hardware = {"torch": str(torch.__version__), "cuda": torch.version.cuda,
            "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
(work / "stage2-hardware.json").write_text(json.dumps(hardware, indent=2))
print(hardware)
assert torch.cuda.is_available(), "Select a GPU T4 accelerator in Kaggle Notebook settings."

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def step_of(directory):
    state = torch.load(directory / "checkpoint.pt", map_location="cpu", weights_only=True)
    if not state["optimizer"]["state"] or state.get("cuda_rng") is None or state.get("cpu_rng") is None:
        raise ValueError("Checkpoint lacks saved Adam or RNG state.")
    return int(state["step"])

first_run = first / relative_run
first_meta = read_json(first_run / "run.json")
stage1 = read_json(first / "stage1-review.json")
assert stage1["complete"] and stage1["step"] == 500 and stage1["run_id"] == source_id
assert first_meta["run_id"] == source_id and first_meta["scene"] == "lego" and first_meta["seed"] == 42
assert first_meta["protocol"] == "dense-vanilla" and first_meta["train_views"] == 100
assert first_meta["data_hash"] == data_hash and first_meta["training_config"]["implementation_hash"] == code_hash
assert first_meta["training_config"]["preset"] == "main" and first_meta["training_config"]["steps"] == 50_000
assert sha(first_run / "checkpoint.pt") == source_digest and step_of(first_run) == 500
observed = subprocess.check_output([sys.executable, "-c",
    "from recovery.training import implementation_hash; print(implementation_hash())"], cwd=repo, text=True).strip()
assert observed == code_hash, "Use the same Stage 0 code kit as the reviewed run."

subprocess.run([sys.executable, "-m", "recovery.data", "--verify", str(first / "manifests/lego.json"),
                "--source", str(repo / "data/blender/lego"), "--output", str(manifest)], cwd=repo, check=True)
assert read_json(manifest)["data_hash"] == data_hash

source_run = first_run
input_step = 500
if partial is not None:
    candidate = partial / relative_run
    saved = read_json(partial / "stage2-review.json")
    assert saved["run_id"] == source_id and saved["source_checkpoint_sha256"] == source_digest
    assert read_json(candidate / "run.json") == first_meta
    assert saved["checkpoint_sha256"] == sha(candidate / "checkpoint.pt")
    input_step = step_of(candidate)
    assert saved["checkpoint_step"] == input_step and 500 <= input_step <= target_step
    source_run = candidate

run.parent.mkdir(parents=True, exist_ok=True)
if not run.exists():
    # Copy into a writable sibling, then publish the complete run in one rename.
    with tempfile.TemporaryDirectory(dir=run.parent) as tmp:
        staged = Path(tmp) / "seed-42"
        staged.mkdir()
        for p in source_run.iterdir():
            if p.is_file() and not p.name.startswith(".") and not p.name.endswith(".tmp"):
                shutil.copyfile(p, staged / p.name)
        assert sha(staged / "checkpoint.pt") == sha(source_run / "checkpoint.pt")
        staged.rename(run)
assert read_json(run / "run.json") == first_meta, "Working run belongs to a different experiment."
ready_step = step_of(run)
assert input_step <= ready_step <= target_step, "Use a fresh session for the selected snapshot."
if ready_step == input_step:
    assert sha(run / "checkpoint.pt") == sha(source_run / "checkpoint.pt")
print(f"Ready at {ready_step} of 50,000 updates, run {source_id}.")

In [ ]:
from IPython.display import display, FileLink
from PIL import Image

start_step = step_of(run)
assert 500 <= start_step <= target_step and read_json(run / "run.json") == first_meta
log_path = work / "dense-resume.log"
cmd = [sys.executable, "-u", "-m", "recovery.training", "--preset", "main", "--protocol", "dense-vanilla",
       "--seed", "42", "--manifest", str(manifest), "--run", str(run), "--device", "cuda:0",
       "--resume", "--save-every", "500", "--stop-at", str(target_step)]
status = dict(stage="dense Lego resume", run_id=source_id, target_step=target_step, complete=False,
              source_checkpoint_sha256=source_digest, start_step=start_step)
started = time.monotonic()
try:
    with log_path.open("a", encoding="utf-8") as log:
        with subprocess.Popen(cmd, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
            for line in proc.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            if proc.wait():
                raise RuntimeError(f"Training stopped with status {proc.returncode}. See dense-resume.log.")
    health = read_json(run / "health.json")
    last = json.loads((run / "training.jsonl").read_text().splitlines()[-1])
    assert step_of(run) == health["step"] == last["step"] == target_step
    with Image.open(run / "train-002000.png") as strip:
        display(strip.resize((576, 192), Image.Resampling.NEAREST))
    status.update(complete=True, steps_per_s=last["steps_per_s"])
    print(f"Saved update {target_step}. Main training averaged {last['steps_per_s']:.2f} updates/s.")
finally:
    status.update(cell_elapsed_seconds=round(time.monotonic() - started, 2),
                  checkpoint_step=step_of(run), checkpoint_sha256=sha(run / "checkpoint.pt"))
    pending = review.with_suffix(".tmp")
    with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_STORED) as z:
        for p in [manifest, work / "stage2-hardware.json", log_path, *sorted(run.rglob("*"))]:
            if p.is_file() and not p.name.startswith(".") and not p.name.endswith(".tmp"):
                z.write(p, p.relative_to(work))
        z.writestr("stage2-review.json", json.dumps(status, indent=2))
    pending.replace(review)
    os.chdir(work.parent)
    print(f"Download {review.name} from Output and send it for review.")
    display(FileLink(review.name))